In [1]:
!pip install -q gradio segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 7.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!ls /content/drive/MyDrive

'Audio from Akhi'	     foodseg_epoch_25.pth
 category_id.txt	     foodseg_epoch_30.pth
'Colab Notebooks'	     foodseg_epoch_35.pth
'Document from Akhi'	     foodseg_epoch_40.pth
 foodseg_best_model.pth      foodseg_epoch_5.pth
 foodseg_best_model_v2.pth  'New Doc 04-30-2025 07.26 (1).pdf'
 foodseg_epoch_10.pth	    'New Doc 04-30-2025 07.26.pdf'
 foodseg_epoch_15.pth	     Verity-By-Colleen-Hoover_copy.pdf
 foodseg_epoch_20.pth


In [4]:
import os
print(os.path.exists("/content/drive/MyDrive/foodseg_best_model_v2.pth"))

True


In [5]:
import torch
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.DeepLabV3Plus(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=104
).to(device)

In [6]:
model_path = "/content/drive/MyDrive/foodseg_best_model_v2.pth"

model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print("Model loaded successfully!")

Model loaded successfully!


In [7]:
# Load category mapping
id_to_name = {}

with open("/content/drive/MyDrive/category_id.txt", "r") as f:
    for line in f:
        parts = line.strip().split()
        class_id = int(parts[0])
        class_name = " ".join(parts[1:])
        id_to_name[class_id] = class_name.lower()

print("Category mapping loaded:", len(id_to_name))

Category mapping loaded: 104


In [8]:
def get_food_group(food_name):
    food_name = food_name.lower()

    # Rice & grains
    if any(word in food_name for word in ["rice", "noodles", "pasta", "bread", "corn", "pie"]):
        return "Grains"

    # Meat
    elif any(word in food_name for word in ["steak", "pork", "chicken", "duck", "sausage", "lamb", "fried meat"]):
        return "Meat"

    # Seafood
    elif any(word in food_name for word in ["fish", "shrimp", "crab", "shellfish"]):
        return "Seafood"

    # Vegetables
    elif any(word in food_name for word in [
        "eggplant","potato","garlic","cauliflower","tomato","kelp","seaweed",
        "spring onion","rape","ginger","okra","lettuce","pumpkin","cucumber",
        "radish","carrot","asparagus","bamboo","broccoli","celery","mint",
        "peas","cabbage","bean sprouts","onion","pepper","beans","mushroom",
        "salad"
    ]):
        return "Vegetables"

    # Fruits
    elif any(word in food_name for word in [
        "apple","date","apricot","avocado","banana","strawberry","cherry",
        "blueberry","raspberry","mango","peach","lemon","pear","fig",
        "pineapple","grape","kiwi","melon","orange","watermelon"
    ]):
        return "Fruits"

    # Nuts & beans
    elif any(word in food_name for word in [
        "almond","cashew","walnut","peanut","red beans","soy"
    ]):
        return "Nuts & Beans"

    # Dairy & drinks
    elif any(word in food_name for word in [
        "milk","cheese","butter","milkshake","coffee","juice","tea","wine"
    ]):
        return "Dairy & Drinks"

    # Snacks & desserts
    elif any(word in food_name for word in [
        "cake","ice cream","pudding","biscuit","chocolate","candy","popcorn","egg tart"
    ]):
        return "Desserts"

    # Fast food
    elif any(word in food_name for word in [
        "pizza","hamburg","fries"
    ]):
        return "Fast Food"

    else:
        return "Other"
group_calories = {
    "Grains": 130,
    "Meat": 250,
    "Seafood": 200,
    "Vegetables": 50,
    "Fruits": 80,
    "Nuts & Beans": 550,
    "Dairy & Drinks": 120,
    "Desserts": 400,
    "Fast Food": 300,
    "Other": 100
}
extra_ingredient_calories = {
    "oil": {"tbsp": 120, "tsp": 40},
    "ghee": {"tbsp": 120, "tsp": 40},
    "butter": {"tbsp": 100, "tsp": 35}
}

In [9]:
import gradio as gr
import cv2
import numpy as np
import torch

# -------- EXTRA INGREDIENT CALORIES --------
extra_ingredient_calories = {
    "oil": {"tbsp": 120, "tsp": 40},
    "ghee": {"tbsp": 120, "tsp": 40},
    "butter": {"tbsp": 100, "tsp": 35}
}

# -------- MAIN FUNCTION --------
def estimate_calories(image, extra_input, weight, age):

    model.eval()

    # Resize image
    image_resized = cv2.resize(image, (320, 320))

    # Noise reduction
    image_resized = cv2.GaussianBlur(image_resized, (3,3), 0)

    # Normalize
    image_norm = image_resized / 255.0

    image_tensor = torch.tensor(
        image_norm,
        dtype=torch.float32
    ).permute(2,0,1).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(image_tensor)
        pred_mask = torch.argmax(output, dim=1).squeeze().cpu().numpy()

    unique_classes = np.unique(pred_mask)
    unique_classes = unique_classes[unique_classes != 0]

    total_pixels = pred_mask.size
    food_pixels = np.sum(pred_mask != 0)

    if food_pixels == 0:
        return image_resized, "No food detected.", ""

    food_ratio = food_pixels / total_pixels

    # Average realistic plate weight
    estimated_plate_weight = 400
    total_food_grams = food_ratio * estimated_plate_weight

    group_mass = {}

    for cls in unique_classes:

        class_pixels = np.sum(pred_mask == cls)

        # remove segmentation glitches
        if class_pixels < 300:
            continue

        class_ratio = class_pixels / food_pixels
        grams = class_ratio * total_food_grams

        food_name = id_to_name.get(cls, "unknown")
        group = get_food_group(food_name)

        group_mass[group] = group_mass.get(group, 0) + grams

    total_calories = 0
    calorie_text = "🍽 Calorie Breakdown\n\n"

    for group, grams in group_mass.items():

        kcal = group_calories[group]
        calories = (grams / 100) * kcal

        total_calories += calories

        calorie_text += f"{group}: {calories:.2f} kcal\n"

    # -------- EXTRA INGREDIENTS --------
    extra_calories = 0

    if extra_input:

        items = extra_input.lower().split(",")

        for item in items:

            words = item.strip().split()

            if len(words) >= 3:

                ingredient = words[0]

                try:
                    quantity = float(words[1])
                except:
                    continue

                unit = words[2]

                if ingredient in extra_ingredient_calories:

                    if unit in extra_ingredient_calories[ingredient]:

                        extra_calories += quantity * extra_ingredient_calories[ingredient][unit]

    if extra_calories > 0:
        calorie_text += f"\nExtra Ingredients: {extra_calories:.2f} kcal"

    total_calories += extra_calories

    calorie_text += "\n---------------------\n"
    calorie_text += f"Meal Calories: {total_calories:.2f} kcal"

    # -------- HEALTH SUGGESTION --------
    suggestion = ""

    if weight and age:

        # Base calorie estimation
        daily_need = weight * 30

        # Age factor
        if age < 18:
            daily_need *= 1.1
        elif age > 40 and age <= 60:
            daily_need *= 0.95
        elif age > 60:
            daily_need *= 0.9

        meal_ratio = total_calories / daily_need

        if meal_ratio < 0.2:
            suggestion = "Light meal. Suitable for weight loss."

        elif meal_ratio < 0.35:
            suggestion = "Balanced meal."

        elif meal_ratio < 0.5:
            suggestion = "High calorie meal."

        else:
            suggestion = "Very high calorie meal. Consider reducing portion."

        suggestion += f"\n\nEstimated Daily Need: {daily_need:.0f} kcal"

    return image_resized, calorie_text, suggestion


# -------- GRADIO UI --------
demo = gr.Interface(
    fn=estimate_calories,
    inputs=[
        gr.Image(type="numpy", label="Upload Food Image"),

        gr.Textbox(
            label="Extra Ingredients (optional)",
            placeholder="Example: oil 1 tbsp, ghee 1 tsp"
        ),

        gr.Number(label="Your Weight (kg)"),

        gr.Number(label="Your Age")
    ],
    outputs=[
        gr.Image(label="Food Image"),

        gr.Textbox(
            label="Calorie Breakdown",
            lines=14
        ),

        gr.Textbox(
            label="Health Suggestion",
            lines=6
        )
    ],
    title="Multi-Dish Food Calorie Estimation",
    description="Upload a food image to estimate calories. Optionally add extra ingredients like oil or ghee.",
    allow_flagging="never"
)


/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


In [10]:
!pkill -f gradio

In [11]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/04/01 09:59:55 [W] [service.go:132] login to server failed: session shutdown


<IPython.core.display.Javascript object>

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> None
